# Coffee Shop Management RAG Bot

## Set Up Local LLM and Embedding Model

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

# 1. Local Embedding Model (Ollama)
Settings.embed_model = OllamaEmbedding(
    model_name="nomic-embed-text"
)

# 2. Local LLM via Ollama (Qwen2.5:3b)
Settings.llm = Ollama(
    model="qwen2.5:3b", request_timeout=120.0, base_url="http://localhost:11434"
)

## Build Query Engines for Each Data Type

In [3]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Load text and pdf files
documents = SimpleDirectoryReader(
    input_files=[
        "./Data/artisan_cafe_knowledge_base.txt",
        "./Data/artisan_cafe_operational_guide.pdf",
    ]
).load_data()

# Build vector store index
doc_index = VectorStoreIndex.from_documents(documents)
doc_query_engine = doc_index.as_query_engine(similarity_top_k=3)

## Tabular Data Engine (Excel File for Orders & Inventory)

In [14]:
import pandas as pd
from llama_index.experimental.query_engine import PandasQueryEngine

# Load Excel sheets into DataFrames
df_orders = pd.read_excel("./Data/artisan_cafe_data.xlsx", sheet_name="Order History")
df_inventory = pd.read_excel(
    "./Data/artisan_cafe_data.xlsx", sheet_name="Inventory Log"
)

# Query engines for Pandas
orders_query_engine = PandasQueryEngine(
    df=df_orders, verbose=True, synthesize_response=True
)
inventory_query_engine = PandasQueryEngine(
    df=df_inventory, verbose=True, synthesize_response=True
)

## Combine into a Multi-Tool Agent / Router Engine

In [15]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool, ToolMetadata

# Define tools
tool_docs = QueryEngineTool(
    query_engine=doc_query_engine,
    metadata=ToolMetadata(
        name="knowledge_base_and_sop",
        description=(
            "Useful for questions regarding coffee recipes, brewing parameters, "
            "opening/closing procedures, barista standards, and supplier contacts."
        ),
    ),
)

tool_orders = QueryEngineTool(
    query_engine=orders_query_engine,
    metadata=ToolMetadata(
        name="order_history",
        description=(
            "Useful for answering questions about customer orders, sales history, "
            "daily revenues, popular items, and order payment methods."
        ),
    ),
)

tool_inventory = QueryEngineTool(
    query_engine=inventory_query_engine,
    metadata=ToolMetadata(
        name="inventory_stock",
        description=(
            "Useful for checking current inventory stock levels, reorder alerts, "
            "unit costs, and item suppliers."
        ),
    ),
)

# Create the Router Engine
router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[tool_docs, tool_orders, tool_inventory],
)

## Test Sample Agent Queries

In [16]:
# Query 1: Recipe / SOP Question (Routes to Document Engine)
response_1 = router_engine.query(
    "How do I prepare a signature Iced Caramel Latte?"
)
print("--- Recipe Query ---")
print(response_1)

# Query 2: Inventory Status (Routes to Inventory Engine)
response_2 = router_engine.query(
    "Which inventory items currently need to be reordered?"
)
print("\n--- Inventory Query ---")
print(response_2)

# Query 3: Order History & Sales Analysis (Routes to Orders Engine)
response_3 = router_engine.query(
    "What was the total revenue generated from Caramel Latte orders?"
)
print("\n--- Orders Query ---")
print(response_3)

--- Recipe Query ---
To prepare a Signature Iced Caramel Latte, follow these steps:

1. Measure and pour 20ml of salted caramel syrup into the bottom of your serving glass.
2. Fill the glass with approximately 150g of ice cubes for adequate cooling.
3. Pour in 180ml of cold whole milk or an alternative milk type, ensuring it's chilled before use to maintain temperature control.
4. Top off by placing a double shot espresso on top of the liquid mixture.
5. Garnish with house caramel sauce across the foam layer atop your drink.

Ensure all ingredients are at the correct temperatures and that the glass is adequately chilled beforehand for an optimal serving experience.
> Pandas Instructions:
```
df[df['Stock Status'].isnull()]
```
> Pandas Output:     Item ID      Category             Item Name  Current Stock    Unit  \
0   INV-001  Coffee Beans  House Espresso Blend           14.5      kg   
1   INV-002         Dairy            Whole Milk           28.0  Liters   
..      ...           ..